##### Creating a parameter map (MERIT)

This notebook visualizes the spatial distribution of learned routing parameters
and predicted channel geometry for the MERIT-Hydro dataset using `GeometryPredictor`.

All 11 output variables from the geometry predictor are plottable:
- **KAN parameters**: `n`, `q_spatial`, `p_spatial`
- **Channel geometry**: `top_width`, `depth`, `bottom_width`, `side_slope`,
  `cross_sectional_area`, `wetted_perimeter`, `hydraulic_radius`, `velocity`

In [ ]:
import logging
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import zarr
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from mpl_toolkits.axes_grid1 import make_axes_locatable

from ddr.geometry import GeometryPredictor

log = logging.getLogger(__name__)

In [ ]:
EXAMPLE_DIR = Path(".")
DATA_DIR = EXAMPLE_DIR / ".." / ".." / "data"

# Load the GeometryPredictor from trained KAN weights
predictor = GeometryPredictor.from_checkpoint(
    checkpoint_path=EXAMPLE_DIR / "ddr-v0.5.2-merit-geometry-weights.pt",
    config_path=EXAMPLE_DIR / "geometry_config.yaml",
    stats_path=DATA_DIR / "statistics" / "merit_attribute_statistics_merit_global_attributes_v2.nc.json",
    device="cpu",
)

In [ ]:
# Load MERIT CONUS reach data (IDs, slope, attributes, discharge proxy)
adjacency = zarr.open_group(DATA_DIR / "merit_conus_adjacency.zarr", mode="r")
comids = adjacency["order"][:]
slope_arr = adjacency["slope"][:]

attrs = xr.open_dataset(DATA_DIR / "merit_global_attributes_v2.nc").sel(COMID=comids)

slope = xr.DataArray(slope_arr, dims="COMID", coords={"COMID": comids})
uparea_km2 = 10 ** attrs["log10_uparea"]
discharge = xr.DataArray(
    (0.01 * uparea_km2).values, dims="COMID", coords={"COMID": comids}
)

print(f"Loaded {len(comids):,} CONUS reaches")

In [ ]:
# Predict all geometry variables
geometry = predictor.predict(attrs, discharge, slope)
geometry

In [ ]:
# Load catchment polygons (NOT linestrings) and join geometry results
catchment_shp = DATA_DIR / "merit" / "cat_pfaf_7_MERIT_Hydro_v07_Basins_v01_bugfix1.shp"
gdf = gpd.read_file(catchment_shp).set_index("COMID")

# Subset to CONUS reaches and attach all predicted variables
gdf = gdf.loc[gdf.index.intersection(comids)]
for var in geometry.data_vars:
    vals = geometry[var].values
    gdf[var] = np.nan
    gdf.loc[comids, var] = vals

gdf = gdf.set_crs(epsg=4326)
print(f"Joined {len(gdf):,} catchment polygons with {len(list(geometry.data_vars))} variables")

In [ ]:
def param_plot(
    gdf: gpd.GeoDataFrame,
    var: str,
    save_name: Path,
    cmap: str = "plasma",
    unit_label: str | None = None,
    title: str | None = None,
    vmin: float | None = None,
    vmax: float | None = None,
    ascending: bool = False,
    dpi: int = 100,
) -> tuple[Figure, Axes]:
    """
    Create a parameter plot for geospatial data with a basemap and colorbar.

    Parameters
    ----------
    gdf : gpd.GeoDataFrame
        GeoDataFrame containing the data to plot.
    var : str
        Column name to visualize.
    save_name : str
        Filename for saving the plot.
    cmap : str, default 'plasma'
        Colormap name for the plot.
    unit_label : str, optional
        Unit label for the colorbar.
    title : str, optional
        Title for the plot.
    vmin : float, optional
        Minimum value for color scaling. If None, uses data minimum.
    vmax : float, optional
        Maximum value for color scaling. If None, uses data maximum.
    ascending : bool, default False
        Whether to sort data in ascending order.
    dpi : int, default 100
        DPI for the figure display.

    Returns
    -------
    tuple of (matplotlib.figure.Figure, matplotlib.axes.Axes)
        Figure and axes objects for further customization.

    """
    if var not in gdf.columns:
        raise KeyError(f"Column '{var}' not found in GeoDataFrame")

    fig, ax = plt.subplots(figsize=(7, 4), dpi=dpi)

    gdf_clean = gdf.dropna(subset=[var])
    if gdf_clean.empty:
        raise ValueError(f"No valid data found for variable '{var}' after dropping NaN values")

    gdf_clean = gdf_clean.sort_values(by=var, ascending=ascending)
    data = gdf_clean[var].values

    if vmin is None:
        vmin = np.min(data)
    if vmax is None:
        vmax = np.nanmax(data)

    gdf_clean.plot(
        ax=ax,
        column=var,
        cmap=cmap,
        linewidth=0.3,
        vmin=vmin,
        vmax=vmax,
        zorder=1,
    )

    cx.add_basemap(
        ax,
        crs=gdf_clean.crs,
        source=cx.providers.CartoDB.Positron,
        alpha=0.6,
        zorder=0,
        attribution=False,
    )

    # CONUS bounds
    ax.set_xlim(-125, -66)
    ax.set_ylim(24, 53)
    ax.set_xticks([])
    ax.set_yticks([])

    if title is not None:
        ax.set_title(title, fontsize=14)

    # Colorbar legend
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.1)
    sm = plt.cm.ScalarMappable(cmap=cmap)
    sm.set_array([])
    sm.set_clim(vmin, vmax)
    cbar = fig.colorbar(sm, cax=cax)

    label_text = var
    if unit_label:
        label_text = f"{var} ({unit_label})"
    cbar.set_label(label_text)

    cbar.formatter.set_powerlimits((-2, 2))
    cbar.update_ticks()

    plt.tight_layout()
    plt.savefig(save_name, dpi=dpi, bbox_inches="tight")

    return fig, ax

In [ ]:
# Plot configuration for all geometry predictor variables
PLOT_DIR = Path("plots")
PLOT_DIR.mkdir(exist_ok=True)

PLOT_CONFIGS = {
    "n": dict(
        title="Manning's Roughness", unit_label="m⁻¹/³s",
        cmap="plasma_r", vmax=0.2, ascending=True,
    ),
    "q_spatial": dict(
        title="Width-Depth Exponent (q)", unit_label="–",
        cmap="viridis", ascending=True,
    ),
    "p_spatial": dict(
        title="Width Coefficient (p)", unit_label="–",
        cmap="viridis", ascending=True,
    ),
    "top_width": dict(
        title="Top Width", unit_label="m",
        cmap="Blues", ascending=True,
    ),
    "depth": dict(
        title="Channel Depth", unit_label="m",
        cmap="Blues", ascending=True,
    ),
    "bottom_width": dict(
        title="Bottom Width", unit_label="m",
        cmap="Blues", ascending=True,
    ),
    "side_slope": dict(
        title="Side Slope", unit_label="–",
        cmap="plasma_r", ascending=True,
    ),
    "cross_sectional_area": dict(
        title="Cross-Sectional Area", unit_label="m²",
        cmap="YlGnBu", ascending=True,
    ),
    "wetted_perimeter": dict(
        title="Wetted Perimeter", unit_label="m",
        cmap="YlGnBu", ascending=True,
    ),
    "hydraulic_radius": dict(
        title="Hydraulic Radius", unit_label="m",
        cmap="YlGnBu", ascending=True,
    ),
    "velocity": dict(
        title="Flow Velocity", unit_label="m/s",
        cmap="coolwarm", ascending=True,
    ),
}

In [ ]:
# Plot a single variable — change the key to plot any of the 11 variables
var = "n"
cfg = PLOT_CONFIGS[var]
param_plot(gdf, var, PLOT_DIR / f"{var}.png", dpi=100, **cfg)

##### Plot all variables at once

Uncomment and run the cell below to generate maps for all 11 geometry predictor outputs.

In [ ]:
for var, cfg in PLOT_CONFIGS.items():
    print(f"Plotting {var}...")
    param_plot(gdf, var, PLOT_DIR / f"{var}.png", dpi=200, **cfg)
    plt.show()